# Fire distribution across Australia

## Accessing Wildfire Data via API

In [14]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import ipywidgets


In [2]:
# 1.
# access api url

## satellite: VIIRS SNPP NRT 
## area: 'world' = entire world 
## day range: '1' = data of one day
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
area_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/world/1' # warum gehen 1, 3 oder 5 tage aber ab 8 oder so nicht mehr??

# 2.
# read in the data from URL

df_area = pd.read_csv(area_url)

# 3.
# have a first glimpse at the data

df_area.head(5)
df_area.shape

(23674, 14)

## Cleaning and Rearranging Data

### Filter for Data only within Australia

In [3]:
# define a bounding box that contains only the area of Australia based on its WGS84 coordinates

coords = [112, -44, 154, -9]

df_aus = df_area[(df_area['longitude'] >= coords[0]) & (df_area['latitude'] >= coords[1]) & (df_area['longitude'] <= coords[2]) & (df_area['latitude'] <= coords[3])].copy()
df_aus.shape
df_aus.head(20)
df_aus.tail()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
23669,-42.74255,146.55779,300.17,0.39,0.36,2026-05-02,1520,N,VIIRS,n,2.0NRT,274.49,1.22,N
23670,-42.06359,147.78604,324.81,0.41,0.37,2026-05-02,1520,N,VIIRS,n,2.0NRT,279.38,4.27,N
23671,-42.06291,147.78107,323.41,0.41,0.37,2026-05-02,1520,N,VIIRS,n,2.0NRT,277.96,4.20,N
23672,-42.06016,147.78691,326.72,0.41,0.37,2026-05-02,1520,N,VIIRS,n,2.0NRT,278.44,2.44,N
23673,-42.05947,147.78192,325.02,0.41,0.37,2026-05-02,1520,N,VIIRS,n,2.0NRT,278.19,3.02,N


### Filter for required Timeframe

In [4]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

df_aus['acq_datetime'] = pd.to_datetime(df_aus['acq_date'] + ' ' + df_aus['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
df_aus.head()

print (f'Australia GMT timezone datetime value range: {df_aus['acq_datetime'].min()} to {df_aus['acq_datetime'].max()}')

Australia GMT timezone datetime value range: 2026-05-02 03:55:00 to 2026-05-02 15:20:00


### Converting raw coordinates into geometries

In [7]:
# the projection EPSG:9473 is used for Australia, as it is recommended for national mapping

# convert latitude, longitude values into point geometry and set crs (since no crs extisting yet) with crs="EPSG:9473" to EPSG:9473

gdf_aus = gpd.GeoDataFrame(
    df_aus, geometry=gpd.points_from_xy(df_aus.longitude, df_aus.latitude), crs="EPSG:9473")
print(gdf_aus.crs)
gdf_aus.head()

EPSG:9473


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acq_datetime,geometry
1031,-43.17596,146.79762,333.68,0.53,0.42,2026-05-02,355,N,VIIRS,n,2.0NRT,277.32,4.22,D,2026-05-02 03:55:00,POINT (146.798 -43.176)
1032,-42.90269,147.86391,338.48,0.48,0.40,2026-05-02,355,N,VIIRS,n,2.0NRT,271.66,6.20,D,2026-05-02 03:55:00,POINT (147.864 -42.903)
1033,-42.87089,147.87529,335.38,0.48,0.40,2026-05-02,355,N,VIIRS,n,2.0NRT,271.61,3.57,D,2026-05-02 03:55:00,POINT (147.875 -42.871)
1034,-42.78834,146.96037,334.01,0.52,0.41,2026-05-02,355,N,VIIRS,n,2.0NRT,281.36,7.59,D,2026-05-02 03:55:00,POINT (146.96 -42.788)
1035,-42.78455,146.95894,351.57,0.52,0.41,2026-05-02,355,N,VIIRS,n,2.0NRT,282.52,7.59,D,2026-05-02 03:55:00,POINT (146.959 -42.785)


## Calculating means etc?

## Visualise it and create interactive Map

In [15]:
aus_map = folium.Map(
    location=[42, 147],
    zoom_start=13,
    tiles="CartoDB Positron",  # A clean, light basemap
)

# Display the map in the notebook
aus_map